 Week 13 — Naive Bayes on Zillow Data
In this notebook, I apply three Naive Bayes variants
(GaussianNB, BernoulliNB, MultinomialNB) to my Zillow
capstone dataset to predict whether a property is
"high value" or not.


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB, BernoulliNB, MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import MinMaxScaler


pd.set_option("display.float_format", lambda x: f"{x:0.4f}")
pd.set_option("display.max_columns", 50)


# ## 1. Load Zillow dataset

df = pd.read_csv("zillow_cleaned.csv")

print(df.shape)
df.head()

(64894, 19)


,bathroomcnt,bedroomcnt,buildingqualitytypeid,calculatedbathnbr,calculatedfinishedsquarefeet,finishedsquarefeet12,fips,fullbathcnt,heatingorsystemtypeid,latitude,longitude,lotsizesquarefeet,propertylandusetypeid,regionidcounty,regionidzip,roomcnt,unitcnt,yearbuilt,taxvaluedollarcnt
0,3.5000,4.0000,6.0000,3.5000,3100.0000,3100.0000,6059.0000,3.0000,7.0000,33634931.0000,-117869207.0000,4506.0000,261.0000,1286.0000,96978.0000,0.0000,1.0000,1998.0000,1023282.0000
1,1.0000,2.0000,4.0000,1.0000,1465.0000,1465.0000,6111.0000,1.0000,2.0000,34449266.0000,-119281531.0000,12647.0000,261.0000,2061.0000,97099.0000,5.0000,1.0000,1967.0000,464000.0000
2,2.0000,3.0000,10.0000,2.0000,1243.0000,1243.0000,6059.0000,2.0000,2.0000,33886168.0000,-117823170.0000,8432.0000,261.0000,1286.0000,97078.0000,6.0000,1.0000,1962.0000,564778.0000
3,3.0000,4.0000,8.0000,3.0000,2376.0000,2376.0000,6037.0000,3.0000,2.0000,34245180.0000,-118240722.0000,13038.0000,261.0000,3101.0000,96330.0000,0.0000,1.0000,1970.0000,145143.0000
4,3.0000,3.0000,8.0000,3.0000,1312.0000,1312.0000,6037.0000,3.0000,2.0000,34185120.0000,-118414640.0000,278581.0000,266.0000,3101.0000,96451.0000,0.0000,1.0000,1964.0000,119407.0000



2. Create a binary target: HighValue
We'll define HighValue = 1 if taxvaluedollarcnt is above
 the 75th percentile, otherwise 0.


In [3]:
target_col = "taxvaluedollarcnt"  # change if your target has a different name

# Drop rows with missing target
df = df.dropna(subset=[target_col])

# Create HighValue based on 75th percentile
threshold = df[target_col].quantile(0.75)
df["HighValue"] = (df[target_col] > threshold).astype(int)

df[["taxvaluedollarcnt", "HighValue"]].head(), threshold

(   taxvaluedollarcnt  HighValue
 0       1023282.0000          1
 1        464000.0000          0
 2        564778.0000          0
 3        145143.0000          0
 4        119407.0000          0,
 572125.75)

3. Select features (numeric only for simplicity)
Naive Bayes in this notebook will use only numeric features.


In [4]:
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()

# Remove the target and any leakage columns from features
cols_to_exclude = ["HighValue", target_col]
feature_cols = [c for c in numeric_cols if c not in cols_to_exclude]

print("Number of numeric features:", len(feature_cols))
feature_cols[:15]
X = df[feature_cols].copy()
y = df["HighValue"].copy()

# For safety, drop rows with missing values in features
data = pd.concat([X, y], axis=1).dropna()
X = data[feature_cols]
y = data["HighValue"]

X.shape, y.value_counts(normalize=True)

Number of numeric features: 18


((64894, 18),
 HighValue
 0   0.7500
 1   0.2500
 Name: proportion, dtype: float64)

4. Train–test split


In [5]:
X_train_num, X_test_num, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train_num.shape, X_test_num.shape

((51915, 18), (12979, 18))

5. Gaussian Naive Bayes
This variant assumes each feature is drawn from a Gaussian
(normal) distribution. It is well-suited for continuous numeric data.

In [6]:
gnb = GaussianNB()
gnb.fit(X_train_num, y_train)

y_pred_g = gnb.predict(X_test_num)

acc_g = accuracy_score(y_test, y_pred_g)
print("GaussianNB accuracy:", acc_g)
print("\nClassification report (GaussianNB):")
print(classification_report(y_test, y_pred_g))

print("Confusion matrix (GaussianNB):")
print(confusion_matrix(y_test, y_pred_g))

GaussianNB accuracy: 0.8152400030819016

Classification report (GaussianNB):
              precision    recall  f1-score   support

           0       0.85      0.92      0.88      9734
           1       0.67      0.52      0.58      3245

    accuracy                           0.82     12979
   macro avg       0.76      0.72      0.73     12979
weighted avg       0.80      0.82      0.81     12979

Confusion matrix (GaussianNB):
[[8908  826]
 [1572 1673]]


6. Bernoulli Naive Bayes
Bernoulli NB expects binary features (0/1).
 We'll binarize each numeric feature:
1 if above the median (in the training set), else 0.

In [7]:
# Compute medians on training data only
medians = X_train_num.median()

X_train_bern = (X_train_num > medians).astype(int)
X_test_bern  = (X_test_num  > medians).astype(int)

X_train_bern.shape, X_test_bern.shape

((51915, 18), (12979, 18))

In [8]:
bnb = BernoulliNB()
bnb.fit(X_train_bern, y_train)

y_pred_b = bnb.predict(X_test_bern)

acc_b = accuracy_score(y_test, y_pred_b)
print("BernoulliNB accuracy:", acc_b)
print("\nClassification report (BernoulliNB):")
print(classification_report(y_test, y_pred_b))

print("Confusion matrix (BernoulliNB):")
print(confusion_matrix(y_test, y_pred_b))

BernoulliNB accuracy: 0.7492102627321057

Classification report (BernoulliNB):
              precision    recall  f1-score   support

           0       0.88      0.77      0.82      9734
           1       0.50      0.68      0.58      3245

    accuracy                           0.75     12979
   macro avg       0.69      0.73      0.70     12979
weighted avg       0.78      0.75      0.76     12979

Confusion matrix (BernoulliNB):
[[7508 2226]
 [1029 2216]]


7. Multinomial Naive Bayes
Multinomial NB works with non-negative count or frequency features.
Here, we rescale numeric features to [0, 1], then treat them as
pseudo-frequencies. This is mainly to illustrate the algorithm.

In [9]:
scaler = MinMaxScaler()

X_train_multi = scaler.fit_transform(X_train_num)
X_test_multi  = scaler.transform(X_test_num)

X_train_multi.shape, X_test_multi.shape
mnb = MultinomialNB()
mnb.fit(X_train_multi, y_train)

y_pred_m = mnb.predict(X_test_multi)

acc_m = accuracy_score(y_test, y_pred_m)
print("MultinomialNB accuracy:", acc_m)
print("\nClassification report (MultinomialNB):")
print(classification_report(y_test, y_pred_m))

print("Confusion matrix (MultinomialNB):")
print(confusion_matrix(y_test, y_pred_m))

MultinomialNB accuracy: 0.7534478773403189

Classification report (MultinomialNB):
              precision    recall  f1-score   support

           0       0.75      1.00      0.86      9734
           1       1.00      0.01      0.03      3245

    accuracy                           0.75     12979
   macro avg       0.88      0.51      0.44     12979
weighted avg       0.81      0.75      0.65     12979

Confusion matrix (MultinomialNB):
[[9734    0]
 [3200   45]]


8. Model comparison

In [10]:
results = pd.DataFrame({
    "Model": ["GaussianNB", "BernoulliNB", "MultinomialNB"],
    "Accuracy": [acc_g, acc_b, acc_m]
})

results

,Model,Accuracy
0,GaussianNB,0.8152
1,BernoulliNB,0.7492
2,MultinomialNB,0.7534


9. In this week’s analysis, I compared three Naive Bayes variants—Gaussian, Bernoulli, and Multinomial—on the task of predicting whether a Zillow property belongs to the top 25% of home values (HighValue = 1). Among the three models, Gaussian Naive Bayes achieved the highest accuracy (0.815), which is expected because the Zillow dataset is dominated by continuous numeric features such as tax value, square footage, lot size, and building characteristics. GaussianNB assumes a normal distribution for each feature, which aligns well with this type of data.

Both Bernoulli and Multinomial Naive Bayes performed moderately (~0.75 accuracy), but they are not ideal fits for Zillow. BernoulliNB requires binary features, so binarizing continuous variables into above/below-median values resulted in a significant loss of information. MultinomialNB expects non-negative count-based features (typical for text frequency data), so even though scaling continuous data to [0,1] allowed the model to run, the feature assumptions were not a natural match.

Overall, the results highlight the importance of choosing a Naive Bayes variant that aligns with the underlying distribution and structure of the features. GaussianNB is the most appropriate choice for continuous real-estate data and therefore delivered the strongest performance. Future improvements could include log-transforming highly skewed variables, adding better-engineered features, or testing Naive Bayes on cluster labels from previous weeks instead of raw home-value thresholds.